# DINOv3 + DPT (CHMv2) on simulated diffraction patterns

Multislice (pyprismatic) simulations in `dinov3/dummy_data/` (SiO2) and `dummy_data/selected_folders/` (AlCo3).
Each `*_training.h5` file is one probe semiangle (1, 2, 4 mrad) × 25 thicknesses:

| dataset | shape | meaning |
|---|---|---|
| `dataMeas` | (256, 256, T) | diffraction pattern at each thickness |
| `dataPots` | (256, 256, T) | potential-derived target (used for ground-truth spot positions) |
| `dataProbe` | (256, 256) | probe aperture in reciprocal space |
| `qx`, `qy` | 256 | reciprocal-space axes (1/Å) |
| `thicknesses` | T | sample thickness (Å) |

The pretrained DINOv3 ViT-L/16 (SAT-493M) + DPT canopy-height head (`dinov3_vitl16_chmv2`) is run unchanged on the
patterns; the question is whether its output can locate Bragg disks. Sections:
1. Setup: data, geometry, ground truth and matching.
2. Raw CHMv2 output on AlCo3.
3. Input scaling: log1p vs true log.
4. Stage 1: geometric disk-centre readout vs no-network baselines (SiO2 and AlCo3).

In [ ]:
from pathlib import Path
import sys
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import maximum_filter
import torch, torch.nn.functional as F

DINOV3_LOCATION  = "/mnt/data/DINOV3/dinov3"
BACKBONE_WEIGHTS = "/mnt/data/DINOV3/weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
DATA_DIR = Path("/mnt/data/DINOV3/dinov3/dummy_data")      # SiO2
IMG       = 256                      # detector px
CROP      = 128                      # central crop in detector px (None = full image); 128 px ≈ ±1.28 1/Å
C0        = (IMG - CROP) // 2 if CROP else 0   # first detector px of the crop
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
sys.path.append(DINOV3_LOCATION)

def crop(img, c=CROP):
    """Central c×c crop of the last two axes (no-op if c is None)."""
    if not c:
        return img
    c0 = (img.shape[-1] - c) // 2
    return img[..., c0:c0 + c, c0:c0 + c]

In [ ]:
# ── Load the SiO2 simulations ───────────────────────────────────────────────
def load_sim(path):
    with h5py.File(path, "r") as h:
        d = {k: h[k][()] for k in h.keys()}
    d["qx"], d["qy"], d["qz"] = d["qx"].ravel(), d["qy"].ravel(), d["qz"].ravel()
    d["uuid"] = path.name.split("_")[0]
    return d

def label(s):
    m = s["meta"]
    return f"{m['pretty_formula']}, α={m['probeSemiangle']} mrad"

meta = pd.read_pickle(DATA_DIR / "simulation_dataframe.pkl")   # pickle only references pandas/numpy/pathlib/uuid
meta.index = meta.index.astype(str)
sims = [load_sim(p) for p in sorted(DATA_DIR.glob("*_training.h5"))]
for s in sims:
    s["meta"] = meta.loc[s["uuid"]]
sims.sort(key=lambda s: s["meta"]["probeSemiangle"])
for s in sims:
    print(f"{s['uuid'][:8]}  {label(s)}  E0={s['meta']['E0']} kV  "
          f"thicknesses={s['thicknesses'].min()}–{s['thicknesses'].max()} Å ({len(s['thicknesses'])})")

## Setup: geometry, ground truth, matching
Ground truth = local maxima of `dataPots` (direct beam excluded). A reflection is **visible** if its disk in `dataMeas`
is ≥ 3× its surrounding ring. Detections are matched greedily within 2 px; recall is reported at 1 and 5 false positives
per image ("vis": invisible lattice points and the direct beam are ignored, not counted as FP).

In [ ]:
# ── Settings, geometry, ground truth, matching ───────────────────────────────
MATCH_PX     = 2.0        # detection ↔ GT matching radius
MIN_DIST_PX  = 4.0        # min distance between detected peaks (SiO2 spot spacing ≈ 8.5 px)
EDGE_PX      = 4          # GT / detections closer than this to the crop edge are not scored
GT_REL       = 0.05       # GT = dataPots maxima ≥ GT_REL × strongest non-direct reflection
GT_NMS       = 5          # local-max window (px) for dataPots; 5 + 5 % keeps the 8.5 px lattice, drops side lobes
VIS_RING     = (1, 3)     # visibility background ring: r+1 … r+3 px around the disk
VIS_CONTRAST = 3.0        # visible: disk ≥ 3 × ring ...
VIS_MIN_REL  = 1e-3       # ... and ≥ 1e-3 × brightest Bragg disk
FP_LEVELS    = (1, 5)     # recall is reported at these false positives per image

LO, HI = C0 + EDGE_PX, C0 + (CROP or IMG) - 1 - EDGE_PX   # scored region (detector px, inclusive)

def probe_geometry(probe):
    """Aperture centroid (row, col) in detector px and radius in px (from its area)."""
    w = (probe > 0.5).astype(float)
    yy, xx = np.mgrid[:probe.shape[0], :probe.shape[1]]
    return np.array([(w * yy).sum(), (w * xx).sum()]) / w.sum(), np.sqrt(w.sum() / np.pi)

def _offsets(r0, r1=None):
    R = int(np.ceil(r1 if r1 else r0)) + 1
    dy, dx = np.mgrid[-R:R + 1, -R:R + 1]; d = np.hypot(dy, dx)
    m = (d <= max(r0, 1.0)) if r1 is None else ((d > r0) & (d <= r1))
    return dy[m], dx[m]

def region_mean(img, centers, r0, r1=None):
    """Mean of img in a disk (radius r0) or ring (r0, r1] around each (row, col) centre."""
    dy, dx = _offsets(r0, r1)
    out = []
    for y, x in np.round(centers).astype(int):
        yy, xx = y + dy, x + dx
        ok = (yy >= 0) & (yy < img.shape[0]) & (xx >= 0) & (xx < img.shape[1])
        out.append(img[yy[ok], xx[ok]].mean())
    return np.asarray(out)

def in_region(pts, lo=LO, hi=HI):
    pts = np.asarray(pts).reshape(-1, 2)
    return np.all((pts >= lo - 0.5) & (pts <= hi + 0.5), axis=1)

def ground_truth(s, t, lo=LO, hi=HI):
    """→ dict(centers (n,2) sub-px, visible (n,), direct (2,)) for file s at thickness index t, inside [lo, hi]."""
    P, M = s["dataPots"][..., t], s["dataMeas"][..., t]
    c0, r = probe_geometry(s["dataProbe"])
    ij = np.argwhere(P == maximum_filter(P, size=GT_NMS))
    ij = ij[np.hypot(*(ij - c0).T) > 1.5]                                  # drop the direct beam
    v = P[tuple(ij.T)]; ij = ij[v >= GT_REL * v.max()]
    cen = []                                                               # sub-px: 3×3 centroid of dataPots
    for y, x in ij:
        y0, y1, x0, x1 = max(y - 1, 0), min(y + 2, P.shape[0]), max(x - 1, 0), min(x + 2, P.shape[1])
        w = P[y0:y1, x0:x1]; yy, xx = np.mgrid[y0:y1, x0:x1]
        cen.append([(w * yy).sum() / w.sum(), (w * xx).sum() / w.sum()])
    cen = np.asarray(cen).reshape(-1, 2)
    disk, ring = region_mean(M, cen, r), region_mean(M, cen, r + VIS_RING[0], r + VIS_RING[1])
    vis = (disk >= VIS_CONTRAST * ring) & (disk >= VIS_MIN_REL * disk.max())
    keep = in_region(cen, lo, hi)
    return dict(centers=cen[keep], visible=vis[keep], direct=c0)

def _greedy(det, targets, ignore, gate):
    used = np.zeros(len(targets), bool)
    status, err = np.zeros(len(det), int), np.full(len(det), np.nan)
    for i, d in enumerate(det):
        dv = np.hypot(*(targets - d).T) if len(targets) else np.array([])
        cand = np.where((dv <= gate) & ~used)[0]
        if len(cand):
            j = cand[np.argmin(dv[cand])]; used[j] = True; status[i], err[i] = 1, dv[j]
        elif len(ignore) and np.hypot(*(ignore - d).T).min() <= gate:
            status[i] = -1
    return status, err

def match(det, gt, gate=MATCH_PX):
    """det sorted by score (desc). Two protocols, status per detection 1 TP / 0 FP / −1 ignored, error px (NaN if not TP):
       "vis": targets = visible spots, invisible lattice points + direct beam ignored;
       "all": targets = every lattice reflection (same labels for every angle), direct beam ignored."""
    c, v, direct = gt["centers"], gt["visible"], gt["direct"][None]
    return dict(zip(("status", "err"), _greedy(det, c[v], np.vstack([c[~v], direct]), gate)),
                **dict(zip(("status_all", "err_all"), _greedy(det, c, direct, gate))))

def froc(rows, proto="vis"):
    """Pool per-image results → (fp per image, recall, score thresholds), score-descending.
    proto "vis": recall of visible spots; "all": recall of all lattice reflections."""
    sk, nk = ("status", "n_vis") if proto == "vis" else ("status_all", "n_all")
    sc = np.concatenate([r["scores"] for r in rows]); st = np.concatenate([r[sk] for r in rows])
    keep = st >= 0; sc, st = sc[keep], st[keep]
    o = np.argsort(-sc); sc, st = sc[o], st[o]
    n_gt, n_img = sum(r[nk] for r in rows), len(rows)
    return np.cumsum(st == 0) / n_img, np.cumsum(st == 1) / max(n_gt, 1), sc

def operating_point(rows, fp_per_img, proto="vis"):
    """Score threshold giving at most fp_per_img false positives per image (pooled over rows), and the recall there."""
    fppi, rec, sc = froc(rows, proto)
    ok = np.where(fppi <= fp_per_img)[0]
    return (sc[ok[-1]], rec[ok[-1]]) if len(ok) else (np.inf, 0.0)

## DINOv3 + DPT (CHMv2) output on AlCo3 diffraction patterns
No spot detection and no training: the pretrained model from `chmv2_inference.ipynb` is run as is on diffraction patterns.
`dinov3_vitl16_chmv2` = frozen DINOv3 ViT-L/16 (SAT-493M) encoder, features from blocks 5, 11, 17 and 23 → DPT decoder trained
to predict **canopy height in metres** (range 0.001–96) from satellite RGB.

Input: each AlCo3 pattern (`selected_folders/0c6670dd-…`) is prepared as (central `CROP` px,
log1p, 1–99.9 % stretch, 3 channels, upsampled to 1024 px), then normalised with the CHMv2 mean/std.
Output: one 1024×1024 map per pattern, shown on a shared log colour scale. The table gives the output range and its
correlation with the log diffraction intensity.

In [ ]:
# ── DINOv3 + DPT (CHMv2) on AlCo3 diffraction patterns: raw model output ──
# Same model as chmv2_inference.ipynb: DINOv3 ViT-L/16 SAT-493M encoder (features from blocks 5, 11, 17, 23)
# + DPT decoder trained for canopy height (output in metres, 0.001–96). No training, no spot detection.
# Needs the setup cells (load_sim, crop, CROP, IMG, DEVICE, label, BACKBONE_WEIGHTS).
from dinov3.hub.depthers import dinov3_vitl16_chmv2

DPT_UUID  = "0c6670dd-6029-42cf-ab7b-cdbc8713c22d"     # AlCo3, Pm-3m (cubic); any folder in selected_folders
DPT_ROOT  = Path("/mnt/data/DINOV3/dinov3/dummy_data/selected_folders")
DPT_HEAD  = "/mnt/data/DINOV3/weights/dinov3_vitl16_chmv2_dpt_head-3703d643.pth"
DPT_SIZE  = 1024                                         # model input px (central CROP upsampled, as in encode())
DPT_T     = [0, 4, 9, 14, 19, 24]                        # thickness indices to show (20 … 500 Å)
CHMV2_MEAN, CHMV2_STD = (0.420, 0.411, 0.296), (0.213, 0.156, 0.143)

if "chmv2_model" not in globals():
    chmv2_model = dinov3_vitl16_chmv2(weights=DPT_HEAD, backbone_weights=BACKBONE_WEIGHTS).to(DEVICE).eval()

dpt_dir  = DPT_ROOT / DPT_UUID
dpt_meta = pd.read_pickle(dpt_dir / "simulation_dataframe.pkl"); dpt_meta.index = dpt_meta.index.astype(str)
dpt_sims = [load_sim(p) for p in sorted(dpt_dir.glob("*_training.h5"))]
for s in dpt_sims:
    s["meta"] = dpt_meta.loc[s["uuid"]]
dpt_sims.sort(key=lambda s: s["meta"]["probeSemiangle"])

def dpt_input(img):
    """Diffraction pattern → CHMv2 input: central CROP, log1p, 1–99.9 % stretch to [0, 1], 3 channels,
    resize to DPT_SIZE, CHMv2 mean/std (the same steps as chmv2_inference.ipynb, after the log scaling)."""
    x = np.log1p(np.clip(crop(img), 0, None)); lo, hi = np.quantile(x, 0.01), np.quantile(x, 0.999)
    x = torch.as_tensor(np.clip((x - lo) / (hi - lo + 1e-6), 0, 1), dtype=torch.float32)[None, None].expand(1, 3, -1, -1)
    x = F.interpolate(x, size=(DPT_SIZE, DPT_SIZE), mode="bilinear", align_corners=False)
    return (x - torch.tensor(CHMV2_MEAN).view(1, 3, 1, 1)) / torch.tensor(CHMV2_STD).view(1, 3, 1, 1)

dpt_out, dpt_stats = {}, []
with torch.inference_mode():
    for fi, s in enumerate(dpt_sims):
        for t in DPT_T:
            pred = chmv2_model(dpt_input(s["dataMeas"][..., t]).to(DEVICE))          # (1, 1, H, W), metres
            pred = F.interpolate(pred, size=(DPT_SIZE, DPT_SIZE), mode="bilinear", align_corners=False)
            out = pred[0, 0].float().cpu().numpy(); dpt_out[fi, t] = out
            logd = np.log10(crop(s["dataMeas"][..., t]) + 1e-8)                      # input brightness, for comparison
            small = F.interpolate(pred.float(), size=logd.shape, mode="area")[0, 0].cpu().numpy()
            dpt_stats.append(dict(mrad=s["meta"]["probeSemiangle"], thickness=s["thicknesses"][t],
                                  out_min=out.min(), out_mean=out.mean(), out_max=out.max(),
                                  corr_with_log_intensity=np.corrcoef(small.ravel(), logd.ravel())[0, 1]))
m0 = dpt_sims[0]["meta"]
print(f"{m0['pretty_formula']} ({m0['space_group_symbol']}): input {DPT_SIZE}×{DPT_SIZE} (central {CROP or IMG} px), "
      f"output {out.shape}, units = CHMv2 canopy height (m)")
display(pd.DataFrame(dpt_stats).set_index(["mrad", "thickness"]))

# Diffraction pattern (log) and DINO + DPT output, one row pair per probe angle
cw = CROP or IMG; c0 = (IMG - cw) // 2
ext_px = [c0 - 0.5, c0 + cw - 0.5, c0 + cw - 0.5, c0 - 0.5]                             # detector px
all_out = np.stack(list(dpt_out.values()))
norm = LogNorm(vmin=all_out.min(), vmax=all_out.max())     # one log colour scale for all maps (values span ~0.01–50 m)
fig, axes = plt.subplots(2 * len(dpt_sims), len(DPT_T), figsize=(3.6 * len(DPT_T), 3.6 * 2 * len(dpt_sims)),
                         squeeze=False, layout="constrained")
for fi, s in enumerate(dpt_sims):
    for k, t in enumerate(DPT_T):
        a, b = axes[2 * fi, k], axes[2 * fi + 1, k]
        a.imshow(np.log10(crop(s["dataMeas"][..., t]) + 1e-8), extent=ext_px, cmap="magma")
        a.set_title(f"{label(s)}, t={s['thicknesses'][t]} Å\ndiffraction (log)", fontsize=8)
        im = b.imshow(dpt_out[fi, t], extent=ext_px, cmap="viridis", norm=norm)
        b.set_title("DINOv3 + DPT (CHMv2) output", fontsize=8)
        for ax in (a, b): ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.4, label="CHMv2 output (canopy height, m, log scale)")
fig.suptitle(f"{m0['pretty_formula']}: frozen DINOv3 + DPT canopy-height decoder applied to diffraction patterns")
plt.show()

### Input scaling: log1p vs true log
`dataMeas` peaks at ~1, and log1p(x) ≈ x for x ≪ 1, so the "log" preprocessing of the previous cell (`dpt_input`)
is almost **linear**: the backbone sees only the brightest disks. The plots look different because they use
`log10(x + 1e-8)`.

**True log:** $\log_{10}(x + \varepsilon \cdot \max x)$ with $\varepsilon = 3 \cdot 10^{-4}$ (`EPS`), mapped to [0, 1] between
the floor $\varepsilon \max x$ and $\max x$ (5 decades). ε sets how much background speckle is lifted.

The cell runs the same DINOv3 + DPT (CHMv2) model on both inputs for AlCo3 (all 25 thicknesses × 3 angles) and reports:
- **reflections_standing_out:** fraction of lattice reflections whose disk is ≥ 0.05 brighter than its surrounding ring
  in the model input (0–1 scale), i.e. how many reflections the backbone can see;
- **dpt_mean_on_disks / off_disks:** mean CHMv2 output on the reflection disks vs everywhere else.

In [ ]:
# ── Input scaling: log1p (current) vs true log, through DINOv3 + DPT (CHMv2) ──
# log1p(x) ≈ x for x ≪ 1, and dataMeas peaks at ~1, so the "log" input used so far is almost linear.
# True log: log10(x + EPS·max x), mapped from [log10(EPS·max), log10(max)] to [0, 1]  → spans 1/EPS (5 decades).
# Needs the setup cells and the previous cell (chmv2_model, dpt_sims, CHMV2_MEAN/STD).
EPS     = 3e-4                         # floor relative to the brightest pixel: smaller = more background speckle
TL_T    = [0, 9, 24]                   # thicknesses shown in the figures (20, 200, 500 Å); stats use all 25
STEP    = 0.05                         # a reflection "stands out" if its disk is ≥ STEP brighter than its ring (input units 0–1)

def true_log01(img, eps=EPS):
    """Central CROP → log10(x + eps·max) scaled to [0, 1] (0 = floor eps·max, 1 = max)."""
    x = np.clip(crop(img), 0, None); m = x.max()
    return np.clip((np.log10(x + eps * m) - np.log10(eps * m)) / (np.log10(m + eps * m) - np.log10(eps * m)), 0, 1)

def log1p01(img):
    """The current preprocessing: log1p, then 1–99.9 % stretch to [0, 1]."""
    x = np.log1p(np.clip(crop(img), 0, None)); lo, hi = np.quantile(x, 0.01), np.quantile(x, 0.999)
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

TL_MODES = {"log1p (current)": log1p01, f"true log, ε={EPS:g}": true_log01}

def to_chmv2(x01):
    x = torch.as_tensor(x01, dtype=torch.float32)[None, None].expand(1, 3, -1, -1)
    x = F.interpolate(x, size=(DPT_SIZE, DPT_SIZE), mode="bilinear", align_corners=False)
    return (x - torch.tensor(CHMV2_MEAN).view(1, 3, 1, 1)) / torch.tensor(CHMV2_STD).view(1, 3, 1, 1)

cw = CROP or IMG; c0 = (IMG - cw) // 2
tl_in, tl_out, tl_rows = {}, {}, []
with torch.inference_mode():
    for fi, s in enumerate(dpt_sims):
        _, r_px = probe_geometry(s["dataProbe"])
        for t in range(len(s["thicknesses"])):
            gt = ground_truth(s, t, lo=c0, hi=c0 + cw - 1)                    # every lattice reflection in the crop
            cen = gt["centers"] - c0
            for mode, fn in TL_MODES.items():
                x01 = fn(s["dataMeas"][..., t]).astype(np.float32)
                pred = chmv2_model(to_chmv2(x01).to(DEVICE)).float()
                out = F.interpolate(pred, size=(cw, cw), mode="area")[0, 0].cpu().numpy()   # back to detector px
                if t in TL_T:
                    tl_in[fi, t, mode] = x01
                    tl_out[fi, t, mode] = F.interpolate(pred, size=(DPT_SIZE, DPT_SIZE), mode="bilinear",
                                                        align_corners=False)[0, 0].cpu().numpy()
                d_in  = region_mean(x01, cen, r_px) - region_mean(x01, cen, r_px + VIS_RING[0], r_px + VIS_RING[1])
                disk  = np.zeros((cw, cw), bool)
                for dy, dx in zip(*_offsets(r_px)):
                    yy, xx = np.round(cen[:, 0]).astype(int) + dy, np.round(cen[:, 1]).astype(int) + dx
                    ok = (yy >= 0) & (yy < cw) & (xx >= 0) & (xx < cw); disk[yy[ok], xx[ok]] = True
                tl_rows.append(dict(mode=mode, mrad=s["meta"]["probeSemiangle"], thickness=s["thicknesses"][t],
                                    n_refl=len(cen), stand_out=(d_in >= STEP).mean(),
                                    out_on_disks=out[disk].mean(), out_off_disks=out[~disk].mean(),
                                    out_max=out.max()))

tl = pd.DataFrame(tl_rows)
summary = tl.groupby(["mode", "mrad"]).agg(
    reflections_standing_out=("stand_out", "mean"),
    dpt_mean_on_disks=("out_on_disks", "mean"), dpt_mean_off_disks=("out_off_disks", "mean"),
    dpt_max=("out_max", "max"))
summary["dpt_on/off"] = summary.dpt_mean_on_disks / summary.dpt_mean_off_disks
print(f"{dpt_sims[0]['meta']['pretty_formula']}, all 25 thicknesses per angle. 'reflections_standing_out' = fraction of "
      f"lattice reflections whose disk is ≥ {STEP} brighter than its ring in the model input (0–1 scale).")
display(summary.unstack("mode").swaplevel(0, 1, axis=1).sort_index(axis=1, level=0, sort_remaining=False))

# What the backbone sees (rows: angle; columns: thickness × mode)
ext_px = [c0 - 0.5, c0 + cw - 0.5, c0 + cw - 0.5, c0 - 0.5]
tl_cols = [(t, m) for t in TL_T for m in TL_MODES]
fig, axes = plt.subplots(len(dpt_sims), len(tl_cols), figsize=(3.3 * len(tl_cols), 3.5 * len(dpt_sims)),
                         squeeze=False, layout="constrained")
for fi, s in enumerate(dpt_sims):
    for ax, (t, m) in zip(axes[fi], tl_cols):
        ax.imshow(tl_in[fi, t, m], extent=ext_px, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{label(s)}, t={s['thicknesses'][t]} Å\n{m}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Model input (before CHMv2 normalisation), 0–1 grey scale"); plt.show()

# DINOv3 + DPT output for each input (same layout, one shared log colour scale)
allv = np.stack(list(tl_out.values())); norm = LogNorm(vmin=allv.min(), vmax=allv.max())
fig, axes = plt.subplots(len(dpt_sims), len(tl_cols), figsize=(3.3 * len(tl_cols), 3.5 * len(dpt_sims)),
                         squeeze=False, layout="constrained")
for fi, s in enumerate(dpt_sims):
    for ax, (t, m) in zip(axes[fi], tl_cols):
        im = ax.imshow(tl_out[fi, t, m], extent=ext_px, cmap="viridis", norm=norm)
        ax.set_title(f"{label(s)}, t={s['thicknesses'][t]} Å\n{m}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.5, label="CHMv2 output (canopy height, m, log scale)")
fig.suptitle("DINOv3 + DPT (CHMv2) output: log1p input vs true-log input"); plt.show()

## Why CHMv2 output fails as a spot locator, and a geometric readout
In the true-log panels every disc is clearly outlined, so the features **see** the discs. The canopy head fails as a
**locator** because it answers a different question ("how tall is the vegetation here?"), so its output has no peak at disc centres.

| # | Observation in the figures above | Cause | Effect on locating spots |
|---|---|---|---|
| 1 | Polarity flips with preprocessing: log1p → discs bright; true log → discs are dark holes | The head scores *texture* as canopy; flat disc interiors read as "ground" | The same disc is a max or a min depending on input, so no fixed peak rule works |
| 2 | No single extremum per disc: under true log each disc is a flat plateau | Output is constant across the disc | Peak picking lands anywhere inside the disc, or on noise |
| 3 | Rims and crescents (log1p, 1 mrad 200–500 Å; 4 mrad outlines) | Edges are texture; interiors are not | Peaks sit on the rim, about one radius from the centre |
| 4 | Background speckle → "canopy" with a low ε floor | ε below the background amplifies speckle | High values off-disc; small discs get buried |
| 5 | Strong scale dependence: 2 mrad discs barely register under log1p | Response depends on disc size after the resize to 1024 | Behaviour changes with angle |
| 6 | Thin samples (20 Å) behave differently | Uniform discs with no internal texture | Same disc looks different at each thickness |

**Stage 1: geometric readout, no training.** A disc in any of these maps is a uniform region with a circular boundary of
known radius r (from the probe). Each pixel is scored by *mean inside radius r − mean in a thin ring just outside*
(r+1 … r+3 px). This peaks at the disc **centre**, works on plateaus (2, 3), and handles either polarity once the sign is set
per map type (1). Four score maps are compared on the same ground truth, matching and FP-per-image scoring as defined in the setup cell:

| Score map | What it tests |
|---|---|
| CHMv2 raw output (log1p) | what failed above |
| CHMv2 true log + disc-ring (−) | do the frozen DINOv3 + DPT features locate discs once the readout is right? |
| Input true log + disc-ring | same readout on the input image: no network |
| Raw input × probe disc (classical p=1) | matched filter on the linear intensity |

How to read it: CHMv2 + disc-ring ≫ CHMv2 raw → the features locate discs and only the readout was wrong.
Input + disc-ring ≈ or > CHMv2 → the network adds nothing on clean simulated discs. A collapse at SiO2 4 mrad is the
disc-overlap limit, not the readout. AlCo3 ≈ SiO2 → no memorisation (this readout uses only geometry).
Caveat: thresholds come from the evaluated set (optimistic); set them on a validation crystal before reporting.

In [ ]:
# ── Stage 1: geometric disc-centre readout on CHMv2 output vs no-network baselines ──
# Needs the setup cells and the two previous code cells. Inference only.
from scipy.signal import fftconvolve

EPS_LOC   = 1e-4      # true-log floor for this test (≈ background level; smaller floors amplify speckle)
MAX_PEAKS = 400       # keep the strongest local maxima per image (far more than 5 FP + all spots)

def disc_kernel(r):
    """Normalised filled disc of radius r (classical matched filter for a probe disc)."""
    R = int(np.ceil(r)) + 1
    yy, xx = np.mgrid[-R:R + 1, -R:R + 1]; k = (np.hypot(yy, xx) <= r).astype(float)
    return k / k.sum()

def disc_ring_kernel(r, gap=1.0, width=2.0):
    """+mean inside radius r, −mean in the ring (r+gap, r+gap+width]: peaks at disc centres."""
    R = int(np.ceil(r + gap + width)) + 1
    yy, xx = np.mgrid[-R:R + 1, -R:R + 1]; d = np.hypot(yy, xx)
    inside, ring = d <= r, (d > r + gap) & (d <= r + gap + width)
    return inside / inside.sum() - ring / ring.sum()

def disc_centre_map(m, r, polarity=+1):
    """Disc-vs-ring contrast of map m (crop px). polarity=-1 when discs are dark in m."""
    return fftconvolve(polarity * m, disc_ring_kernel(r), mode="same")

def pick_peaks(score, min_dist=MIN_DIST_PX, max_peaks=MAX_PEAKS):
    """Local maxima of the robust z-scored map, parabolic sub-px refinement → (n, 2) crop px, scores (desc)."""
    med = np.median(score); z = (score - med) / (1.4826 * np.median(np.abs(score - med)) + 1e-12)
    rad = max(1, int(round(min_dist)))
    ij = np.argwhere(z == maximum_filter(z, size=2 * rad + 1, mode="nearest"))
    sc = z[tuple(ij.T)]; o = np.argsort(-sc)[:max_peaks]; ij, sc = ij[o], sc[o]
    pos = ij.astype(float)
    for a in (0, 1):
        lo_, hi_ = ij.copy(), ij.copy(); lo_[:, a] -= 1; hi_[:, a] += 1
        ok = (lo_[:, a] >= 0) & (hi_[:, a] < z.shape[a])
        l = z[tuple(np.clip(lo_, 0, z.shape[a] - 1).T)]; rr = z[tuple(np.clip(hi_, 0, z.shape[a] - 1).T)]
        den = l - 2 * sc + rr
        pos[:, a] += np.clip(np.where(ok & (den < 0), 0.5 * (l - rr) / np.where(den < 0, den, -1), 0), -0.5, 0.5)
    return pos, sc

def score_maps(s, t, r):
    raw  = crop(s["dataMeas"][..., t]).astype(np.float64)
    x_tl = true_log01(s["dataMeas"][..., t], EPS_LOC).astype(np.float32)
    x_lp = log1p01(s["dataMeas"][..., t]).astype(np.float32)
    with torch.inference_mode():
        chm = lambda x: F.interpolate(chmv2_model(to_chmv2(x).to(DEVICE)).float(),
                                      size=(cw, cw), mode="area")[0, 0].cpu().numpy()
        out_tl, out_lp = chm(x_tl), chm(x_lp)
    return {
        "CHMv2 raw output (log1p)":           out_lp,
        "CHMv2 true log + disc-ring (−)":     disc_centre_map(np.log10(np.clip(out_tl, 0, None) + 1e-2), r, -1),
        "Input true log + disc-ring":         disc_centre_map(x_tl, r, +1),
        "Raw input × probe disc (classical)": fftconvolve(raw, disc_kernel(r), mode="same"),
    }

loc_rows = []
for crystal, sims_ in (("SiO2", sims), ("AlCo3", dpt_sims)):
    for s in sims_:
        _, r = probe_geometry(s["dataProbe"]); mr = s["meta"]["probeSemiangle"]
        for t in range(len(s["thicknesses"])):
            gt = ground_truth(s, t)
            for name, m in score_maps(s, t, r).items():
                det, sc = pick_peaks(m); det = det + c0                    # crop px → detector px
                keep = in_region(det); det, sc = det[keep], sc[keep]
                loc_rows.append(dict(crystal=crystal, method=name, mrad=mr, t_idx=t, det=det, scores=sc,
                                     **match(det, gt), n_vis=int(gt["visible"].sum()), n_all=len(gt["centers"])))

loc_tab = []
for (cr, m, mr), g in pd.DataFrame(loc_rows).groupby(["crystal", "method", "mrad"], sort=False):
    g = g.to_dict("records")
    loc_tab.append(dict(crystal=cr, method=m, mrad=mr,
                        vis_recall_1FP=operating_point(g, 1, "vis")[1], vis_recall_5FP=operating_point(g, 5, "vis")[1]))
loc_tab = pd.DataFrame(loc_tab)
print("Recall of visible spots at 1 / 5 false positives per image, all 25 thicknesses, thresholds picked on the "
      "evaluated set (optimistic).")
display(loc_tab.pivot_table(index=["crystal", "method"], columns="mrad", values=["vis_recall_1FP", "vis_recall_5FP"],
                            sort=False).round(3))

### Stage 1 results
Recall of **visible** spots at 1 / 5 FP per image (all 25 thicknesses; thresholds picked on the evaluated set, so optimistic):

| Score map | SiO2 1 / 2 / 4 mrad (5 FP) | AlCo3 1 / 2 / 4 mrad (5 FP) |
|---|---|---|
| CHMv2 raw output (log1p) | 0.31 / 0.00 / 0.00 | 0.30 / 0.00 / 0.00 |
| CHMv2 true log + disc-ring (−) | 0.16 / 0.01 / 0.07 | 0.38 / 0.03 / 0.19 |
| **Input true log + disc-ring** (no network) | **0.96 / 0.94 / 0.50** | **0.94 / 0.86 / 0.76** |
| Raw input × probe disc (classical) | 0.65 / 0.71 / 0.14 | 0.93 / 0.80 / 0.55 |

- **The geometric readout does not rescue the CHMv2 output.** CHMv2 + disc-ring is no better than the raw CHMv2 output on SiO2
  and only slightly better on AlCo3; both stay near zero at 2 and 4 mrad. The canopy head throws away the disc-centre
  information, so its output map is not a usable spot score with any fixed readout.
- **The no-network baseline is strong.** The same disc-ring filter applied to the true-log input finds 86–96 % of visible spots
  at 1–2 mrad on both crystals and beats the classical linear matched filter everywhere, most at 4 mrad. The true-log
  scaling (ε = 1e-4) is what makes weak discs usable.
- **Similar recall on SiO2 and AlCo3** for the geometric readouts: no crystal-specific tuning is involved.
- SiO2 4 mrad stays lowest for every method: disc overlap (diameter ≈ 2.4× spot spacing), not a readout problem.

**Implications.** This tests the CHMv2 *canopy head*, not the frozen DINOv3 features: the stage 1 outcome says the head is
the bottleneck, not whether the features carry centre information. Stage 2 (a heatmap + offset head trained on the frozen
features over several crystals, crystal-split train/val/test, threshold set on validation) answers that, and its bar is now
**"Input true log + disc-ring"**: ≈ 0.94–0.96 recall at 5 FP at 1 mrad, 0.86–0.94 at 2 mrad, 0.50–0.76 at 4 mrad.
The polarity of the CHMv2 map (−1, discs dark under true log) was set from the figures above and not tuned.